# Predicción de Riesgo Crediticio usando Redes Bayesianas

Este proyecto demuestra el aprendizaje de estructura de Redes Bayesianas usando el algoritmo Hill-Climbing e inferencia probabilística para evaluación de riesgo crediticio.

## Tabla de Contenidos
1. [Setup](#1-setup)
2. [Marco Teórico](#2-marco-teórico)
3. [Carga de Datos](#3-carga-de-datos)
4. [Aprendizaje de Estructura](#4-aprendizaje-de-estructura)
5. [Refinamiento de la Red](#5-refinamiento-de-la-red)
6. [Inferencia Probabilística](#6-inferencia-probabilística)
7. [Conclusiones](#7-conclusiones)

## 1. Setup

In [ ]:
"""Setup: Importar librerías y configurar el entorno."""

# Librerías de terceros
# Suprimir warnings
import warnings

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from pgmpy.estimators import BicScore, HillClimbSearch, MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination
from pgmpy.models import DiscreteBayesianNetwork

warnings.filterwarnings('ignore')

# Constantes
ARCHIVO_DATOS = "data/Riesgo_Credito.csv"
TAMANO_FIGURA = (12, 10)
TAMANO_NODO = 5000
COLOR_NODO = "#90e2ec"

print("Setup completado.")

## 2. Marco Teórico

### 2.1 Aprendizaje de Estructura de Redes Bayesianas

Cuando el conocimiento experto sobre las dependencias entre variables no está disponible, podemos **aprender** la estructura de la red a partir de los datos.

### 2.2 Algoritmo Hill-Climbing

**Hill-Climbing** es un algoritmo de búsqueda local que maximiza una función de puntuación modificando iterativamente la estructura de la red:

**Operaciones:**
- Agregar aristas entre variables
- Eliminar aristas existentes
- Invertir dirección de aristas

**Funciones de Puntuación:**
- **AIC** (Criterio de Información de Akaike)
- **BIC** (Criterio de Información Bayesiano)
- **BDe** (Equivalencia Bayesiana de Dirichlet)

**Nota:** Encontrar la estructura óptima es NP-HARD; Hill-Climbing proporciona una buena aproximación pero no garantiza el óptimo global.

### 2.3 Estimación de Máxima Verosimilitud

Una vez aprendida la estructura, estimamos las Distribuciones de Probabilidad Condicional (CPDs) usando Estimación de Máxima Verosimilitud a partir de las frecuencias observadas en los datos.

## 3. Carga de Datos

### 3.1 Descripción del Dataset

El dataset de riesgo crediticio contiene información sobre solicitantes de préstamos:

| Variable | Descripción |
|----------|-------------|
| Evaluacion crediticia historica | Evaluación crediticia histórica (0-4) |
| Edad | Rango de edad |
| Genero | Género |
| Evaluacion Sueldo | Evaluación salarial |
| Residencia | Tipo de residencia |
| Nivel de ahorro | Nivel de ahorro |
| Monto del credito | Monto del crédito |
| Duracion | Duración del préstamo |
| Proposito | Propósito del préstamo |
| Riesgo | Nivel de riesgo (good/bad) |

In [ ]:
def cargar_datos_credito(ruta_archivo: str) -> pd.DataFrame:
    """Carga dataset de riesgo crediticio.

    Args:
        ruta_archivo: Ruta al archivo CSV.

    Returns:
        DataFrame con datos de crédito.
    """
    return pd.read_csv(ruta_archivo, usecols=range(1, 11))

In [ ]:
# Cargar dataset
datos = cargar_datos_credito(ARCHIVO_DATOS)
print(f"Forma del dataset: {datos.shape}")
datos.head()

## 4. Aprendizaje de Estructura

### 4.1 Hill-Climbing con Puntuación BIC

In [ ]:
def aprender_estructura(datos: pd.DataFrame) -> nx.DiGraph:
    """Aprende estructura de Red Bayesiana usando Hill-Climbing.

    Args:
        datos: DataFrame con observaciones.

    Returns:
        Estructura de red aprendida.
    """
    metodo_score = BicScore(data=datos)
    hc = HillClimbSearch(data=datos, scoring_method=metodo_score)
    return hc.estimate()

In [ ]:
# Aprender estructura de la red
modelo_aprendido = aprender_estructura(datos)
print(f"Aristas aprendidas: {len(modelo_aprendido.edges())}")

In [ ]:
def visualizar_red(modelo: nx.DiGraph, titulo: str = "Estructura de Red") -> None:
    """Visualiza estructura de Red Bayesiana.

    Args:
        modelo: Red a visualizar.
        titulo: Título del gráfico.
    """
    pos = {
        "Evaluacion crediticia historica": [4, 1],
        "Edad": [3, 6],
        "Genero": [4, 4],
        "Evaluacion Sueldo": [2, 2],
        "Residencia": [3, 5],
        "Nivel de ahorro": [3, -1],
        "Monto del credito": [3, 3],
        "Duracion": [4, 2],
        "Proposito": [1, 2],
        "Riesgo": [3, 0],
    }

    plt.figure(figsize=TAMANO_FIGURA)
    nx.draw(
        modelo, pos,
        with_labels=True,
        font_weight="bold",
        node_size=TAMANO_NODO,
        node_color=COLOR_NODO,
        font_size=8,
        arrows=True
    )
    plt.title(titulo, fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
visualizar_red(modelo_aprendido, "Estructura Aprendida (Hill-Climbing + BIC)")

## 5. Refinamiento de la Red

### 5.1 Correcciones con Conocimiento del Dominio

La estructura aprendida captura correlaciones pero puede tener direcciones de causalidad invertidas. Basado en conocimiento del dominio:

**Correcciones necesarias:**
1. **Nivel de Ahorro → Riesgo** (no Riesgo → Ahorro): El nivel de ahorro causa la evaluación de riesgo
2. **Sueldo → Monto del Crédito** (no al revés): Mayor sueldo permite préstamos más grandes
3. **Género → Residencia** (no al revés): Basado en patrones históricos de propiedad
4. **Propósito → Monto del Crédito** (no al revés): El propósito determina el monto requerido
5. **Historial Crediticio → Duración** (no al revés): Buen historial permite plazos más largos

In [ ]:
def refinar_estructura(modelo: nx.DiGraph) -> list:
    """Aplica correcciones de conocimiento del dominio a la estructura aprendida.

    Args:
        modelo: Estructura de red aprendida.

    Returns:
        Lista de aristas corregidas.
    """
    refinado = modelo.copy()

    # Aplicar correcciones de dirección de aristas
    correcciones = [
        ("Riesgo", "Nivel de ahorro", "Nivel de ahorro", "Riesgo"),
        ("Monto del credito", "Evaluacion Sueldo", "Evaluacion Sueldo", "Monto del credito"),
        ("Residencia", "Genero", "Genero", "Residencia"),
        ("Monto del credito", "Proposito", "Proposito", "Monto del credito"),
        ("Duracion", "Evaluacion crediticia historica", "Evaluacion crediticia historica", "Duracion"),
    ]

    for ant_desde, ant_hasta, nuevo_desde, nuevo_hasta in correcciones:
        if refinado.has_edge(ant_desde, ant_hasta):
            refinado.remove_edge(ant_desde, ant_hasta)
            refinado.add_edge(nuevo_desde, nuevo_hasta)

    return list(refinado.edges())

In [ ]:
# Refinar estructura y crear Red Bayesiana
aristas_refinadas = refinar_estructura(modelo_aprendido)
red_bayesiana = DiscreteBayesianNetwork(aristas_refinadas)

print("Aristas de la red refinada:")
for arista in red_bayesiana.edges():
    print(f"  {arista[0]} → {arista[1]}")

In [ ]:
# Visualizar red refinada
pos_refinada = {
    "Evaluacion crediticia historica": [4, 3],
    "Edad": [3, 7],
    "Genero": [4, 6],
    "Evaluacion Sueldo": [2, 5],
    "Residencia": [3, 5],
    "Nivel de ahorro": [2, 1],
    "Monto del credito": [3, 3],
    "Duracion": [4, 1],
    "Proposito": [1, 5],
    "Riesgo": [3, 0],
}

plt.figure(figsize=TAMANO_FIGURA)
nx.draw(
    red_bayesiana, pos_refinada,
    with_labels=True,
    font_weight="bold",
    node_size=TAMANO_NODO,
    node_color=COLOR_NODO,
    font_size=8,
    arrows=True
)
plt.title("Estructura de Red Bayesiana Refinada", fontsize=14)
plt.tight_layout()
plt.show()

### 5.2 Aprendizaje de Parámetros

In [ ]:
# Ajustar CPDs usando Estimación de Máxima Verosimilitud
red_bayesiana.fit(data=datos, estimator=MaximumLikelihoodEstimator)

# Validar modelo
es_valido = red_bayesiana.check_model()
print(f"Validación del modelo: {'Aprobado' if es_valido else 'Fallido'}")

## 6. Inferencia Probabilística

### 6.1 Propiedades de Independencia

In [ ]:
# Independencias locales para la variable Riesgo
print("Independencias Locales para Riesgo:")
print(red_bayesiana.local_independencies("Riesgo"))

In [ ]:
# Contar independencias markovianas globales para Riesgo
independencias_riesgo = [
    ind for ind in red_bayesiana.get_independencies().get_assertions()
    if "Riesgo" in next(iter(ind.get_assertion()[0]))
]
print(f"\nTotal de independencias markovianas globales para Riesgo: {len(independencias_riesgo)}")

### 6.2 Consultas de Inferencia

In [ ]:
# Crear motor de inferencia
inferencia = VariableElimination(red_bayesiana)

#### Consulta 1: P(Riesgo=bad | Edad=21)

In [ ]:
# Consulta: Probabilidad de riesgo para cliente joven (edad 21)
resultado_edad = inferencia.query(["Riesgo"], evidence={"Edad": "19 to 28"})
prob_malo_joven = resultado_edad.get_value(Riesgo="bad")

print(f"P(Riesgo=bad | Edad=19-28): {prob_malo_joven:.4f}")
print(f"P(Riesgo=good | Edad=19-28): {1 - prob_malo_joven:.4f}")

#### Consulta 2: P(Riesgo | Evaluación Crediticia Histórica)

In [ ]:
# Tabla de probabilidad condicional para Riesgo dado Historial Crediticio
probs_riesgo = {"bad": [], "good": []}

for calificacion in range(5):
    resultado = inferencia.query(
        ["Riesgo"],
        evidence={"Evaluacion crediticia historica": calificacion}
    )
    probs_riesgo["bad"].append(resultado.get_value(Riesgo="bad"))
    probs_riesgo["good"].append(resultado.get_value(Riesgo="good"))

# Mostrar como tabla
tabla_prob_cond = pd.DataFrame(
    probs_riesgo,
    index=[f"Calificación={i}" for i in range(5)]
).T

print("P(Riesgo | Evaluación Crediticia Histórica):")
tabla_prob_cond

#### Consulta 3: P(Riesgo=bad | Duración=36 meses)

In [ ]:
# Consulta: Riesgo para préstamos a largo plazo (3 años)
resultado_duracion = inferencia.query(["Riesgo"], evidence={"Duracion": "24 to 72"})
prob_malo_largo = resultado_duracion.get_value(Riesgo="bad")

print(f"P(Riesgo=bad | Duración=24-72 meses): {prob_malo_largo:.4f}")
print(f"P(Riesgo=good | Duración=24-72 meses): {1 - prob_malo_largo:.4f}")

## 7. Conclusiones

### Hallazgos Principales

1. **Aprendizaje de Estructura:** Hill-Climbing con puntuación BIC aprendió exitosamente una estructura de red capturando dependencias entre variables, aunque algunas direcciones de aristas requirieron correcciones con conocimiento del dominio.

2. **Factores de Riesgo:**
   - Clientes jóvenes (19-28) tienen ~31% de probabilidad de riesgo alto
   - Mal historial crediticio (calificación 0-1) aumenta significativamente el riesgo (~60-65%)
   - Préstamos a largo plazo (24-72 meses) tienen ~52% de probabilidad de riesgo alto

3. **Impacto del Historial Crediticio:** La probabilidad de riesgo malo disminuye sustancialmente a medida que mejora la evaluación crediticia histórica (de 64.5% en calificación 0 a 18.8% en calificación 4).

### Notas Técnicas

- El aprendizaje de estructura es NP-HARD; Hill-Climbing proporciona soluciones aproximadas
- El conocimiento del dominio es esencial para corregir direcciones de aristas aprendidas
- Eliminación de Variables permite inferencia exacta eficiente en el modelo aprendido